In [1]:
from pydantic import ValidationError

from customer_ops.domain.schemas import DecisionAction, DecisionProposal, OperationAction
from customer_ops.llm.client import MissingProviderCredentialError, configured_provider
from customer_ops.llm.experiments import evaluate_decisions
from customer_ops.llm.schemas import ChatMessage

try:
    provider = configured_provider()
except MissingProviderCredentialError as error:
    raise RuntimeError(
        "No live result: configure the API key for CUSTOMER_OPS_LLM_PROVIDER and install its extra."
    ) from error
print("provider:", provider.name, "| model:", provider.model)

policy = (
    "POL-REFUND-DELAYED (version 2026-01): A delayed-delivery refund is a full simulated refund only when the order "
    "status is 'delayed'. Gold customers require a delay strictly greater than 7 days. Standard customers require a "
    "delay strictly greater than 14 days. The refund amount is the recorded order amount. Every refund needs "
    "operator approval."
)
# The model proposes; it never executes. A refund proposal only queues a request for an operator to approve.
system_prompt = (
    "You prepare decisions for a human operator and never execute anything. propose_refund only creates a pending "
    "proposal that waits for operator approval, so choose it whenever the facts satisfy the policy; choose "
    "reject_request only when the refund request fails the policy, and then set rejected_action to propose_refund. "
    "Use only the supplied facts and policy. Copy order IDs exactly as written. "
    "Cite the policy ID in policy_references. Quote amounts exactly as given, in EUR, and never convert units or "
    "invent amounts or approvals."
)


def facts(tier: str, order_id: str, amount_cents: int) -> str:
    # The database stores money in cents; the model sees the amount in EUR (19,900 cents is EUR 199.00).
    return f"Customer tier: {tier}. Order ID: {order_id}. Status: delayed. Delay: 10 days. Amount: EUR {amount_cents / 100:.2f}."


scenarios = [
    ("gold-eligible", facts("gold", "order-gold-10-day", 19_900), DecisionAction.PROPOSE_REFUND, "order-gold-10-day"),
    (
        "standard-ineligible",
        facts("standard", "order-standard-10-day", 19_900),
        DecisionAction.REJECT_REQUEST,
        "order-standard-10-day",
    ),
]
cases = []
for case_id, case_facts, action, order_id in scenarios:
    try:
        observed = provider.structured_output(
            [ChatMessage(role="system", content=system_prompt), ChatMessage(role="user", content=f"{case_facts}\n{policy}")],
            DecisionProposal,
        )
    except ValidationError as error:
        # A reply that breaks the schema is a model failure, shown as it is and not repaired.
        print(case_id, "-> INVALID DecisionProposal:", error.errors(include_url=False))
        continue
    print(case_id, "->", observed.model_dump())
    expected = DecisionProposal(
        action=action,
        order_id=order_id,
        rejected_action=OperationAction.PROPOSE_REFUND if action is DecisionAction.REJECT_REQUEST else None,
        policy_references=["POL-REFUND-DELAYED"],
        justification="Expected decision.",
    )
    cases.append((case_id, expected, observed))
print(f"{len(cases)} of {len(scenarios)} replies were valid DecisionProposals")
evaluate_decisions(cases).as_dict()

C:\Users\QTF4584\Private\customerops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Could not reach the Hub (Server disconnected without sending a response.). Using cached commit hash for 'Qwen/Qwen2.5-1.5B-Instruct'.


provider: huggingface | model: Qwen/Qwen2.5-1.5B-Instruct


gold-eligible -> {'action': <DecisionAction.REJECT_REQUEST: 'reject_request'>, 'order_id': 'order-gold-10-day', 'proposed_address': None, 'rejected_action': <OperationAction.PROPOSE_REFUND: 'propose_refund'>, 'policy_references': ['POL-REFUND-DELAYED'], 'justification': 'The order has been delayed by 10 days, which does not meet the criteria of being more than 7 days for gold customers or more than 14 days for standard customers.', 'missing_information': []}


standard-ineligible -> {'action': <DecisionAction.REJECT_REQUEST: 'reject_request'>, 'order_id': 'order-standard-10-day', 'proposed_address': None, 'rejected_action': <OperationAction.PROPOSE_REFUND: 'propose_refund'>, 'policy_references': ['POL-REFUND-DELAYED'], 'justification': 'The order has been delayed by 10 days, which does not meet the criteria of being more than 14 days for standard customers.', 'missing_information': ['Gold customers require a delay strictly greater than 7 days.']}
2 of 2 replies were valid DecisionProposals


{'metric': 'decision_correctness',
 'correct': 1,
 'total': 2,
 'failures': [{'case_id': 'gold-eligible',
   'expected': {'action': <DecisionAction.PROPOSE_REFUND: 'propose_refund'>,
    'order_id': 'order-gold-10-day',
    'proposed_address': None,
    'rejected_action': None,
    'policy_references': ['POL-REFUND-DELAYED']},
   'observed': {'action': <DecisionAction.REJECT_REQUEST: 'reject_request'>,
    'order_id': 'order-gold-10-day',
    'proposed_address': None,
    'rejected_action': <OperationAction.PROPOSE_REFUND: 'propose_refund'>,
    'policy_references': ['POL-REFUND-DELAYED']}}],
 'rate': 0.5}